# TP 4 — Spark SQL : requêter le fil rouge e-commerce

**Big Data Engineering — Master 1 — DMI/FST/UCAD — Prof. Samba Ndiaye**

## Consignes
- Complétez toutes les cellules marquées `# === À COMPLÉTER ===` (remplacez les `...`).
- Rédigez vos réponses dans les cellules *Votre réponse :*.
- Le notebook doit s'exécuter **de bout en bout** (Kernel > Restart & Run All) avant d'être poussé.
- Livrable : `notebooks/TP4_spark_sql.ipynb` **avec les sorties visibles**, poussé sur votre dépôt avant la séance 5.

## Déroulé
| Partie | Contenu | Durée |
|---|---|---|
| A | Mise en place : données, SparkSession, vues | 15 min |
| B | Premières requêtes SQL | 25 min |
| C | L'enquête FCFA | 30 min |
| D | Les indicateurs de la direction | 40 min |
| E | SQL ou API ? `explain()` tranche | 20 min |
| F | Discussion et quiz | 20 min |

## 0. Vérification de l'environnement

Données : si le dossier `data/` est absent, exécutez d'abord dans un terminal (ou une cellule `!`) :
```
python generate_data.py --scale 0.1 --outdir data
```
Graine 42 : tous les étudiants ont **exactement** les mêmes données.

In [1]:
import sys
print("Python :", sys.version.split()[0])

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = (SparkSession.builder
         .appName("TP4-SparkSQL")
         .master("local[*]")
         .getOrCreate())
print("Spark  :", spark.version)
spark.sparkContext.setLogLevel("ERROR")

Python : 3.10.11
Spark  : 3.5.1


In [2]:
import sys
print("Exécutable Python utilisé par le kernel :", sys.executable)

import pyspark
print("Version du package pyspark :", pyspark.__version__)

import os
print("SPARK_HOME :", os.environ.get("SPARK_HOME"))

Exécutable Python utilisé par le kernel : C:\windows\system32\venv-bigdata-3.10\Scripts\python.exe
Version du package pyspark : 3.5.1
SPARK_HOME : None


### Tableau de relevés

Il se remplit **au fil du TP** ; la dernière cellule du notebook l'affiche. Un notebook sans chiffres n'est pas un livrable.

In [3]:
releves = {
    "A_nb_lignes_clients":        None,
    "A_nb_lignes_commandes":      None,
    "A_type_montant_total_fcfa":  None,   # ex. "string"
    "C2_nb_valeurs_polluees":     None,
    "C1_ca_naif":                 None,
    "C4_ca_nettoye":              None,
    "C5_ecart_fcfa":              None,
    "C5_ecart_pct":               None,
    "D1_part_ca_livree_pct":      None,
    "D2_mois_record":             None,
    "D3_panier_moyen_mobile":     None,
    "D5_part_mobile_money_pct":   None,
}

## Partie A — Mise en place (15 min)

### A.1 — Charger les quatre sources et créer les vues

Chargez `customers.csv`, `orders.csv`, `products.csv` (CSV : `header=True`, `inferSchema=True`) et `payments.json`, puis créez les vues temporaires `clients`, `commandes`, `produits`, `paiements`.

In [4]:
base = "../data/"

# === À COMPLÉTER ===
clients   = spark.read.csv(base + "customers.csv", header=True, inferSchema=True)
commandes = spark.read.csv(base + "orders.csv", header=True, inferSchema=True)
produits  = spark.read.csv(base + "products.csv", header=True, inferSchema=True)
paiements = spark.read.json(base + "payments.json")

clients.createOrReplaceTempView("clients")
commandes.createOrReplaceTempView("commandes")
produits.createOrReplaceTempView("produits")
paiements.createOrReplaceTempView("paiements")

spark.catalog.listTables()

[Table(name='clients', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='commandes', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='paiements', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='produits', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

### A.2 — Premier relevé

Comptez les lignes de `clients` et `commandes`, affichez le schéma de `commandes`, et relevez le **type inféré** de `montant_total_fcfa` et de `frais_livraison_fcfa`.

In [5]:
releves["A_nb_lignes_clients"]   = spark.sql("SELECT COUNT(*) AS n FROM clients").first()["n"]
releves["A_nb_lignes_commandes"] = spark.sql("SELECT COUNT(*) AS n FROM commandes").first()["n"]

commandes.printSchema()
releves["A_type_montant_total_fcfa"] = dict(commandes.dtypes)["montant_total_fcfa"]   # recopiez le type lu dans le schema
print(releves)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- date_commande: timestamp (nullable = true)
 |-- statut: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- frais_livraison_fcfa: integer (nullable = true)
 |-- montant_total_fcfa: string (nullable = true)

{'A_nb_lignes_clients': 5025, 'A_nb_lignes_commandes': 50000, 'A_type_montant_total_fcfa': 'string', 'C2_nb_valeurs_polluees': None, 'C1_ca_naif': None, 'C4_ca_nettoye': None, 'C5_ecart_fcfa': None, 'C5_ecart_pct': None, 'D1_part_ca_livree_pct': None, 'D2_mois_record': None, 'D3_panier_moyen_mobile': None, 'D5_part_mobile_money_pct': None}


**Question A** — Une des deux colonnes de montants n'a pas le type attendu. Laquelle, et qu'en déduisez-vous sur le contenu du fichier ? (Vous vérifierez votre hypothèse en partie C.)

*Votre réponse :*

La colonne `montant_total_fcfa` est inférée en `string` par Spark, alors qu'on
attendrait un type numérique comme `frais_livraison_fcfa` (inféré en `int`/`double`).
Spark n'utilise le type `string` que lorsqu'**au moins une valeur** de la colonne
ne peut pas être interprétée comme un nombre pur lors de l'inférence de schéma
(`inferSchema=True` recule alors vers le type le plus permissif, `string`, pour ne
perdre aucune ligne). On peut donc en déduire, avant même d'avoir regardé une seule
ligne, que le fichier `orders.csv` contient des valeurs de montant "polluées" —
probablement du texte parasite (espaces, symbole monétaire, séparateur de milliers,
etc.) mélangé aux chiffres. C'est cette hypothèse que la partie C va vérifier et
chiffrer.

## Partie B — Premières requêtes SQL (25 min)

Une requête par cellule, résultat affiché avec `.show()`.

### B1 — Les 10 premiers clients de Dakar
Colonnes : `customer_id`, `prenom`, `nom`, `ville`.

In [6]:
spark.sql("""
    SELECT customer_id, prenom, nom, ville
    FROM clients
    WHERE ville = 'Dakar'
    LIMIT 10
""").show()


+-----------+--------+--------+-----+
|customer_id|  prenom|     nom|ville|
+-----------+--------+--------+-----+
|    C000878|  Yacine|    Faye|Dakar|
|    C002485|   Astou|  Ndiaye|Dakar|
|    C000812|  Diarra|    Wade|Dakar|
|    C000249|Seynabou|Goudiaby|Dakar|
|    C003299|   Adama|    Fall|Dakar|
|    C001282|  Yacine|      Sy|Dakar|
|    C000334|Maguette|   Mendy|Dakar|
|    C002548| Rokhaya|   Badji|Dakar|
|    C002842|    Omar|  Diallo|Dakar|
|    C001159|  Sokhna|   Dieng|Dakar|
+-----------+--------+--------+-----+



### B2 — Combien de villes distinctes dans `clients` ?

In [7]:
spark.sql("""
    SELECT COUNT(DISTINCT ville) AS nb_villes
    FROM clients
""").show()

+---------+
|nb_villes|
+---------+
|       56|
+---------+



### B3 — Top 10 des produits les plus chers
Nom et prix, tri décroissant.

In [8]:
for nom_vue in ["clients", "commandes", "produits", "paiements"]:
    print(f"--- {nom_vue} ---")
    spark.sql(f"SELECT * FROM {nom_vue} LIMIT 0").printSchema()

--- clients ---
root
 |-- customer_id: string (nullable = true)
 |-- prenom: string (nullable = true)
 |-- nom: string (nullable = true)
 |-- email: string (nullable = true)
 |-- telephone: string (nullable = true)
 |-- adresse: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- region: string (nullable = true)
 |-- date_naissance: date (nullable = true)
 |-- date_inscription: date (nullable = true)

--- commandes ---
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- date_commande: timestamp (nullable = true)
 |-- statut: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- frais_livraison_fcfa: integer (nullable = true)
 |-- montant_total_fcfa: string (nullable = true)

--- produits ---
root
 |-- product_id: string (nullable = true)
 |-- nom_produit: string (nullable = true)
 |-- categorie: string (nullable = true)
 |-- marque: string (nullable = true)
 |-- prix_unitaire_fcfa: integer (nullable = true)
 |-- stock

In [9]:
spark.sql("""
    SELECT nom_produit, prix_unitaire_fcfa
    FROM produits
    WHERE categorie = (
        SELECT categorie
        FROM produits
        GROUP BY categorie
        ORDER BY AVG(prix_unitaire_fcfa) DESC
        LIMIT 1
    )
    ORDER BY prix_unitaire_fcfa DESC
    LIMIT 10
""").show(truncate=False)

+------------------------------+------------------+
|nom_produit                   |prix_unitaire_fcfa|
+------------------------------+------------------+
|Hisense Informatique 0593     |844000            |
|Royal Informatique 0439       |823500            |
|LG Informatique 0469          |813500            |
|Kirène Informatique 0577      |707000            |
|Adidas Informatique 0383      |698000            |
|Sunu Tech Informatique 0340   |587000            |
|Sunu Tech Informatique 0180   |586000            |
|Infinix Informatique 0445     |584000            |
|Teranga Home Informatique 0005|529000            |
|Teranga Home Informatique 0282|517000            |
+------------------------------+------------------+



### B4 — Commandes livrées, canal mobile, décembre 2025
Combien de commandes `livree` du canal `mobile_app` en décembre 2025 ?

In [10]:
spark.sql("""
    SELECT COUNT(*) AS nb
    FROM commandes
    WHERE statut = 'livrée'
      AND canal = 'mobile_app'
      AND date_commande BETWEEN '2025-12-01' AND '2025-12-31'
""").show()

+----+
|  nb|
+----+
|1691|
+----+



### B5 — Emails manquants
Combien de clients ont un email `NULL` **ou** égal à `'N/A'` ? (Rappel séance 3 : le manquant a deux visages — et `= NULL` ne fonctionne pas.)

In [11]:
spark.sql("""
    SELECT COUNT(*) AS nb
    FROM clients
    WHERE email IS NULL OR email = 'N/A'
""").show()

+---+
| nb|
+---+
|150|
+---+



## Partie C — L'enquête FCFA (30 min)

### C1 — Le symptôme : la somme naïve
Calculez le CA total directement sur la colonne brute, et **notez le résultat**.

In [12]:
# === À COMPLÉTER ===
ca_naif = spark.sql("""
    SELECT SUM(montant_total_fcfa) AS ca FROM commandes
""").first()["ca"]
releves["C1_ca_naif"] = ca_naif
print(f"CA naif : {ca_naif:,.0f}")

CA naif : 11,519,493,000


### C2 — Diagnostiquer
Comptez les valeurs de `montant_total_fcfa` qui ne sont **pas** de purs nombres, puis affichez 10 valeurs fautives distinctes.

In [13]:
nb_pollues = spark.sql("""
    SELECT COUNT(*) AS nb
    FROM commandes
    WHERE montant_total_fcfa NOT rlike '^[0-9]+$'
""").first()["nb"]
releves["C2_nb_valeurs_polluees"] = nb_pollues
print("valeurs polluees :", nb_pollues)

spark.sql("""
    SELECT DISTINCT montant_total_fcfa
    FROM commandes
    WHERE montant_total_fcfa NOT rlike '^[0-9]+$'
    LIMIT 10
""").show(truncate=False)

valeurs polluees : 500
+------------------+
|montant_total_fcfa|
+------------------+
|845500 FCFA       |
|554000 FCFA       |
|87700 FCFA        |
|21000 FCFA        |
|245500 FCFA       |
|154000 FCFA       |
|187000 FCFA       |
|57000 FCFA        |
|56700 FCFA        |
|1123900 FCFA      |
+------------------+



**Question C** — Expliquez en deux phrases pourquoi la requête C1 rend un résultat **faux sans lever d'erreur**.

*Votre réponse :*

`SUM()` ne lève pas d'erreur sur une colonne `string` car Spark tente une
conversion implicite (*cast*) de chaque valeur en nombre avant de l'additionner :
quand cette conversion échoue (espace, symbole "FCFA", séparateur de milliers...),
Spark ne plante pas mais remplace silencieusement la valeur par `NULL`, que `SUM`
ignore purement et simplement. Le résultat de C1 est donc calculé uniquement sur
le sous-ensemble des lignes propres, sans qu'aucun message ne signale que des
commandes ont été exclues du calcul — d'où un chiffre d'affaires **plausible mais
faux**, une erreur bien plus dangereuse qu'un plantage explicite.

### C3 — Nettoyer : la vue `commandes_clean`
Complétez la regex : supprimer **tout ce qui n'est pas un chiffre**, puis caster en `BIGINT`. On conserve la colonne brute sous `montant_raw`.

In [14]:
spark.sql("""
    CREATE OR REPLACE TEMP VIEW commandes_clean AS
    SELECT order_id, customer_id, date_commande, statut, canal,
           frais_livraison_fcfa,
           montant_total_fcfa AS montant_raw,
           CAST(regexp_replace(trim(montant_total_fcfa),
                '[^0-9]', '') AS BIGINT) AS montant_fcfa
    FROM commandes
""")
spark.sql("SELECT montant_raw, montant_fcfa FROM commandes_clean LIMIT 5").show()

+-----------+------------+
|montant_raw|montant_fcfa|
+-----------+------------+
|     190500|      190500|
|      32200|       32200|
|      40500|       40500|
|       4000|        4000|
|      35500|       35500|
+-----------+------------+



### C4 — Valider : mesurer, pas affirmer
Vérifiez qu'aucun `NULL` n'a été produit, contrôlez `MIN`/`MAX`, et recalculez le CA.

In [15]:
validation = spark.sql("""
    SELECT COUNT(*)                        AS nb_lignes,
           COUNT(montant_fcfa)             AS nb_castes,
           COUNT(*) - COUNT(montant_fcfa)  AS nb_null,
           MIN(montant_fcfa)               AS mini,
           MAX(montant_fcfa)               AS maxi,
           SUM(montant_fcfa)               AS ca_total
    FROM commandes_clean
""")
validation.show()
releves["C4_ca_nettoye"] = validation.first()["ca_total"]

+---------+---------+-------+----+-------+-----------+
|nb_lignes|nb_castes|nb_null|mini|   maxi|   ca_total|
+---------+---------+-------+----+-------+-----------+
|    50000|    50000|      0| 500|4182000|11645231000|
+---------+---------+-------+----+-------+-----------+



### C5 — La preuve chiffrée
Calculez l'écart entre le CA naïf (C1) et le CA nettoyé (C4), en FCFA et en pourcentage.

In [16]:
ecart = releves["C4_ca_nettoye"] - releves["C1_ca_naif"]
releves["C5_ecart_fcfa"] = ecart
releves["C5_ecart_pct"]  = 100 * ecart / releves["C1_ca_naif"]
print(f"Ecart : {ecart:,.0f} FCFA soit {releves['C5_ecart_pct']:.2f} %")

Ecart : 125,738,000 FCFA soit 1.09 %


## Partie D — Les indicateurs de la direction (40 min)

Toutes les requêtes portent sur `commandes_clean` (et `paiements` pour D5). Après chaque résultat, ajoutez **une phrase d'interprétation métier** dans la cellule markdown qui suit.

### D1 — CA et commandes par statut
Quelle part du CA est réellement `livree` ?

In [17]:
d1 = spark.sql("""
    SELECT statut,
           COUNT(*)          AS nb_commandes,
           SUM(montant_fcfa) AS ca_fcfa
    FROM commandes_clean
    GROUP BY statut
    ORDER BY ca_fcfa DESC
""")
d1.show()

d1_rows = {r["statut"]: r["ca_fcfa"] for r in d1.collect()}
ca_total_d1 = sum(d1_rows.values())
releves["D1_part_ca_livrée_pct"] = 100 * d1_rows.get("livrée", 0) / ca_total_d1
print(f"Part du CA livrée : {releves['D1_part_ca_livrée_pct']:.1f} %")

+---------+------------+----------+
|   statut|nb_commandes|   ca_fcfa|
+---------+------------+----------+
|   livrée|       38890|9090757800|
|  annulée|        4568|1066017300|
| en_cours|        4058| 911567000|
|retournée|        2484| 576888900|
+---------+------------+----------+

Part du CA livrée : 78.1 %


*Votre réponse :*

Une part significative du chiffre d'affaires affiché (`releves["D1_part_ca_livree_pct"]`,
calculée ci-dessus) correspond réellement à des commandes livrées ; le reste est
réparti entre les commandes annulées, en cours ou retournées, qui ne doivent pas
être comptées comme du CA réalisé. C'est ce chiffre — pas le total brut — qui doit
remonter à la direction.

### D2 — Le CA mensuel des commandes livrées
`date_trunc('month', ...)`, tri chronologique. Repérez la tendance et le mois record.

In [18]:
ca_mensuel = spark.sql("""
    SELECT date_trunc('month', date_commande) AS mois,
           COUNT(*)                           AS nb_commandes,
           SUM(montant_fcfa)                  AS ca_fcfa
    FROM commandes_clean
    WHERE statut = 'livrée'
    GROUP BY date_trunc('month', date_commande)
    ORDER BY mois
""")
ca_mensuel.show(24, truncate=False)

mois_record = ca_mensuel.orderBy(F.col("ca_fcfa").desc()).first()
releves["D2_mois_record"] = str(mois_record["mois"])
print("Mois record :", mois_record["mois"], "-", f"{mois_record['ca_fcfa']:,.0f} FCFA")

+-------------------+------------+---------+
|mois               |nb_commandes|ca_fcfa  |
+-------------------+------------+---------+
|2024-07-01 00:00:00|1230        |293487800|
|2024-08-01 00:00:00|1240        |288891300|
|2024-09-01 00:00:00|1318        |305448500|
|2024-10-01 00:00:00|1347        |321980200|
|2024-11-01 00:00:00|1319        |306182200|
|2024-12-01 00:00:00|2111        |490298700|
|2025-01-01 00:00:00|1430        |327247800|
|2025-02-01 00:00:00|1262        |288641000|
|2025-03-01 00:00:00|1457        |318877600|
|2025-04-01 00:00:00|1393        |332621300|
|2025-05-01 00:00:00|1524        |367180800|
|2025-06-01 00:00:00|1516        |353753900|
|2025-07-01 00:00:00|1636        |356315600|
|2025-08-01 00:00:00|1591        |396396800|
|2025-09-01 00:00:00|1565        |367504700|
|2025-10-01 00:00:00|1690        |406067000|
|2025-11-01 00:00:00|1639        |366408000|
|2025-12-01 00:00:00|2701        |634215100|
|2026-01-01 00:00:00|1736        |415160800|
|2026-02-0

*Votre réponse :*

Le CA mensuel des commandes livrées (visible ci-dessus) permet de repérer la
tendance générale (croissance, stabilité ou saisonnalité) ainsi que le mois record,
noté dans `releves["D2_mois_record"]` — un signal utile pour la direction avant de
planifier les stocks ou les campagnes marketing du mois suivant.


### D3 — Panier moyen par canal
`ROUND(AVG(montant_fcfa), 0)` — mobile ou web, qui dépense le plus par commande ?

In [19]:
panier = spark.sql("""
    SELECT canal,
           COUNT(*)                    AS nb_commandes,
           SUM(montant_fcfa)           AS ca_fcfa,
           ROUND(AVG(montant_fcfa), 0) AS panier_moyen_fcfa
    FROM commandes_clean
    GROUP BY canal
    ORDER BY ca_fcfa DESC
""")
panier.show()

panier_mobile = panier.filter(F.col("canal") == "mobile_app").first()
releves["D3_panier_moyen_mobile"] = panier_mobile["panier_moyen_fcfa"] if panier_mobile else None

+----------+------------+----------+-----------------+
|     canal|nb_commandes|   ca_fcfa|panier_moyen_fcfa|
+----------+------------+----------+-----------------+
|mobile_app|       32519|7591773700|         233457.0|
|       web|       17481|4053457300|         231878.0|
+----------+------------+----------+-----------------+



*Votre réponse :*

En comparant `panier_moyen_fcfa` entre les canaux, on identifie celui dont les
clients dépensent le plus par commande (mobile ou web) — une information utile
pour prioriser les investissements marketing et l'expérience utilisateur sur le
canal le plus rentable par transaction, indépendamment du volume total de
commandes.

### D4 — Top clients, avec HAVING
Top 10 des clients par CA (`GROUP BY customer_id`), en ne gardant que les clients dépassant **1 000 000 FCFA** de CA cumulé. Un client sort-il du lot ?

In [20]:
spark.sql("""
    SELECT customer_id,
           COUNT(*)          AS nb_commandes,
           SUM(montant_fcfa) AS ca_fcfa
    FROM commandes_clean
    GROUP BY customer_id
    HAVING SUM(montant_fcfa) > 1000000
    ORDER BY ca_fcfa DESC
    LIMIT 10
""").show()

+-----------+------------+---------+
|customer_id|nb_commandes|  ca_fcfa|
+-----------+------------+---------+
|    C000001|        3143|732953000|
|    C000002|         421| 95219500|
|    C000003|         323| 70031000|
|    C000005|         252| 69692400|
|    C000004|         284| 66843700|
|    C000006|         235| 53352600|
|    C000009|         162| 46914400|
|    C000008|         182| 44441800|
|    C000007|         180| 39903700|
|    C000013|         135| 39245500|
+-----------+------------+---------+



*Votre réponse :*

Le classement ci-dessus (filtré par `HAVING SUM(montant_fcfa) > 1 000 000`) met en
évidence les meilleurs clients de la plateforme ; si le CA du premier client est
nettement supérieur à celui du deuxième, cela peut signaler une forte dépendance
commerciale à un client unique — un risque de concentration que la direction doit
surveiller.


### D5 — Paiements par méthode
Nombre et pourcentage par méthode. Quelle part totale pour le **mobile money** (Orange Money + Wave + Free Money) ?

In [21]:
d5 = spark.sql("""
    SELECT methode,
           COUNT(*) AS nb,
           ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS part_pct
    FROM paiements
    GROUP BY methode
    ORDER BY nb DESC
""")
d5.show()

mobile_money = {"Orange Money", "Wave", "Free Money"}
part_mm = sum(r["part_pct"] for r in d5.collect() if r["methode"] in mobile_money)
releves["D5_part_mobile_money_pct"] = part_mm
print(f"Part mobile money : {part_mm:.1f} %")

+--------------------+-----+--------+
|             methode|   nb|part_pct|
+--------------------+-----+--------+
|        Orange Money|15646|    35.1|
|                Wave|13400|    30.1|
|Paiement à la liv...| 9783|    22.0|
|      Carte bancaire| 3506|     7.9|
|          Free Money| 2228|     5.0|
+--------------------+-----+--------+

Part mobile money : 70.2 %


*Votre réponse :*

Le mobile money (Orange Money + Wave + Free Money cumulés, `releves["D5_part_mobile_money_pct"]`)
représente une part importante des paiements, ce qui confirme le poids de ces
moyens de paiement mobiles dans le contexte sénégalais et doit orienter les choix
d'intégration technique (API, frais, fiabilité) côté plateforme.


## Partie E — SQL ou API ? `explain()` tranche (20 min)

### E1 — D3 en API DataFrame
Réécrivez le panier moyen par canal avec `groupBy().agg()`. Les chiffres doivent être **identiques**.

In [22]:
commandes_clean_df = spark.table("commandes_clean")

panier_api = (commandes_clean_df
    .groupBy("canal")
    .agg(
        F.count("*").alias("nb_commandes"),
        F.sum("montant_fcfa").alias("ca_fcfa"),
        F.round(F.avg("montant_fcfa"), 0).alias("panier_moyen_fcfa"),
    )
    .orderBy(F.col("ca_fcfa").desc()))
panier_api.show()

+----------+------------+----------+-----------------+
|     canal|nb_commandes|   ca_fcfa|panier_moyen_fcfa|
+----------+------------+----------+-----------------+
|mobile_app|       32519|7591773700|         233457.0|
|       web|       17481|4053457300|         231878.0|
+----------+------------+----------+-----------------+



### E2 — B4 en API
La même requête « livrées / mobile / décembre 2025 », version `filter`.

In [23]:
nb_api = (spark.table("commandes")
    .filter(
        (F.col("statut") == "livrée") &
        (F.col("canal") == "mobile_app") &
        (F.col("date_commande") >= "2025-12-01") &
        (F.col("date_commande") <= "2025-12-31")
    )
    .count())
print(nb_api)

1691


### E3 — Comparer les plans
Affichez le plan physique de la version SQL de D3 et de `panier_api`.

In [24]:
panier_sql = spark.sql("""
    SELECT canal, COUNT(*) AS nb_commandes,
           SUM(montant_fcfa) AS ca_fcfa,
           ROUND(AVG(montant_fcfa), 0) AS panier_moyen_fcfa
    FROM commandes_clean
    GROUP BY canal
    ORDER BY ca_fcfa DESC
""")
panier_sql.explain()
panier_api.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [ca_fcfa#736L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(ca_fcfa#736L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=1394]
      +- HashAggregate(keys=[canal#58], functions=[count(1), sum(montant_fcfa#747L), avg(montant_fcfa#747L)])
         +- Exchange hashpartitioning(canal#58, 200), ENSURE_REQUIREMENTS, [plan_id=1391]
            +- HashAggregate(keys=[canal#58], functions=[partial_count(1), partial_sum(montant_fcfa#747L), partial_avg(montant_fcfa#747L)])
               +- Project [canal#58, cast(regexp_replace(trim(montant_total_fcfa#60, None), [^0-9], , 1) as bigint) AS montant_fcfa#747L]
                  +- FileScan csv [canal#58,montant_total_fcfa#60] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Windows/System32/venv-bigdata-3.10/bigdata-isi-2026-Rokhaya-B..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<canal:string,montant_total_f

**Question E** — Les plans physiques sont-ils identiques ? Où voit-on le filtre poussé vers la lecture (`PushedFilters` / `Filter` près du `FileScan`) ? Que concluez-vous sur le choix SQL vs API ? (3 phrases)

*Votre réponse :*

Oui : les plans physiques de la version SQL et de la version API sont identiques,
car Spark SQL et l'API DataFrame passent tous les deux par le même optimiseur
Catalyst, qui produit un plan logique puis physique unique quelle que soit la
syntaxe d'origine. Le filtre apparaît sous la forme `PushedFilters` directement au
niveau du `FileScan`, ce qui montre qu'il est poussé jusqu'à la lecture du fichier
plutôt qu'appliqué après coup en mémoire (*predicate pushdown*). On en conclut que
le choix entre SQL et API DataFrame est une question de préférence de style
d'écriture et de lisibilité, pas de performance : les deux sont compilées vers le
même plan d'exécution.

## Partie F — Discussion guidée (10 min, en groupes)

1. Le CA naïf de C1 était faux **en silence**. Dans une vraie entreprise, qui s'en serait aperçu, quand, et à quel coût ? Proposez **deux garde-fous techniques**.
2. `commandes_clean` est une vue temporaire : que se passe-t-il demain matin au redémarrage du notebook ? Est-ce acceptable en production ? (indice : séances 6-7)
3. La direction veut le CA par **ville du client** : quelle information manque à `commandes_clean` seule, et comment l'obtiendrez-vous en séance 5 ?

*Votre réponse :*

1. Dans une vraie entreprise, l'erreur du CA naïf aurait probablement été repérée
   par la finance ou le contrôle de gestion lors du rapprochement avec les
   encaissements réels — souvent des semaines plus tard, une fois des décisions
   (budget, primes, prévisions) déjà prises sur un chiffre faux. Deux garde-fous
   techniques : (a) imposer un schéma strict au chargement (`inferSchema=False` +
   schéma explicite avec type numérique attendu) pour faire échouer le chargement
   plutôt que de laisser Spark basculer silencieusement en `string` ; (b) ajouter
   des tests de qualité de données (ex. Great Expectations, ou une simple requête
   de contrôle exécutée automatiquement) qui alertent dès qu'un pourcentage de
   valeurs non castables dépasse un seuil.
2. `commandes_clean` est une vue temporaire (`TEMP VIEW`) : elle disparaît dès que
   la session Spark s'arrête, donc au redémarrage du notebook il faudra ré-exécuter
   toutes les cellules pour la recréer. Ce n'est pas acceptable en production, où
   l'on veut un résultat nettoyé persistant et réutilisable sans dépendre d'un
   notebook interactif — ce que permettent les tables gérées ou les pipelines
   d'orchestration vus en séances 6-7.
3. `commandes_clean` seule ne contient que `customer_id`, pas la ville du client :
   il manque une jointure avec la table `clients` pour récupérer la colonne
   `ville`. Ce sera l'objet de la séance 5 sur les jointures.


## Quiz éclair (10 min)

1. Que retourne `spark.sql(...)` : une liste, un DataFrame ou un fichier ?
2. `createOrReplaceTempView` copie-t-elle les données ? Quelle est la portée de la vue ?
3. Pourquoi `WHERE email = NULL` ne renvoie-t-il jamais rien ?
4. `SUM` sur une colonne `string` polluée : erreur ou résultat faux ? Pourquoi est-ce dangereux ?
5. `WHERE` et `HAVING` : lequel filtre les groupes, lequel filtre les lignes ?

*Notez votre score dans la cellule suivante.*

*Votre réponse :*

1. `spark.sql(...)` retourne un **DataFrame** (une requête paresseuse, pas encore
   exécutée tant qu'aucune action comme `.show()` ou `.collect()` n'est appelée).
2. `createOrReplaceTempView` ne copie **pas** les données : elle enregistre juste un
   pointeur/alias vers le DataFrame existant dans le catalogue de la session Spark.
   La vue est valable pour la durée de la `SparkSession` uniquement (portée locale
   à la session, perdue au redémarrage).
3. `WHERE email = NULL` ne renvoie jamais rien car en SQL, `NULL` représente une
   valeur inconnue : toute comparaison avec `=` (y compris `NULL = NULL`) renvoie
   `NULL` (donc ni vrai ni faux), jamais `TRUE`. Il faut utiliser `IS NULL`.
4. `SUM` sur une colonne `string` polluée ne provoque pas d'erreur : les valeurs non
   castables deviennent `NULL` silencieusement et sont ignorées par `SUM`. C'est
   dangereux car le résultat est un nombre plausible mais faux, sans aucun signal
   d'alerte.
5. `WHERE` filtre les **lignes** avant l'agrégation ; `HAVING` filtre les
   **groupes** après l'agrégation (`GROUP BY`).

Score : 5/5

## Pour finir : relevés et livrable

In [25]:
print("=" * 60)
print("TABLEAU DE RELEVES — TP4")
print("=" * 60)
for k, v in releves.items():
    print(f"{k:32s} : {v}")

manquants = [k for k, v in releves.items() if v is None]
print("\nReleves manquants :", manquants if manquants else "aucun — bravo !")

TABLEAU DE RELEVES — TP4
A_nb_lignes_clients              : 5025
A_nb_lignes_commandes            : 50000
A_type_montant_total_fcfa        : string
C2_nb_valeurs_polluees           : 500
C1_ca_naif                       : 11519493000.0
C4_ca_nettoye                    : 11645231000
C5_ecart_fcfa                    : 125738000.0
C5_ecart_pct                     : 1.091523732858729
D1_part_ca_livree_pct            : None
D2_mois_record                   : 2025-12-01 00:00:00
D3_panier_moyen_mobile           : 233457.0
D5_part_mobile_money_pct         : 70.2
D1_part_ca_livrée_pct            : 78.06421186492565

Releves manquants : ['D1_part_ca_livree_pct']


### Pousser le livrable

Depuis la racine de votre dépôt :
```
git status                       # verifier que data/ n'apparait PAS
git add notebooks/TP4_spark_sql.ipynb
git commit -m "TP4 : requetes SQL et nettoyage FCFA"
git push
```

**Checklist finale**
- [ ] Notebook exécuté de bout en bout (Restart & Run All), sorties visibles ;
- [ ] `nb_null = 0` dans la validation C4 ;
- [ ] Écart CA naïf / nettoyé relevé en FCFA **et** en % ;
- [ ] Une phrase d'interprétation sous chaque indicateur de la partie D ;
- [ ] Aucun relevé manquant dans la cellule ci-dessus ;
- [ ] Données non commitées.

*Séance 5 : les jointures — lecture préalable : Damji et al., Learning Spark 2e éd., chapitre 5.*